# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_multicolor


In [1]:
# ONNX dependency setup.
import importlib.util, subprocess, sys
required = {'onnx':'onnx', 'onnxruntime':'onnxruntime', 'torch':'torch'}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import onnx, onnxruntime as ort
print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.5 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [2]:
import json, hashlib, zipfile, random, sys
from pathlib import Path
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort


In [3]:



ROOT=Path.cwd()
TASK_CANDIDATES=[
    Path('/kaggle/input/competitions/neurogolf-2026/task145.json'),
    Path('/mnt/data/task145.json'),
    Path('/mnt/data/task145(1).json'),
    ROOT/'upload'/'task145(1).json',
]
TASK=next((p for p in TASK_CANDIDATES if p.exists()),None)
assert TASK is not None,f'task145 JSON not found: {TASK_CANDIDATES}'
OUT=ROOT/'task145_v1_component_rank' 
OUT.mkdir(exist_ok=True)
MODEL=OUT/'task145.onnx'

In [4]:
nodes=[]
def tensor(name,value,dtype=np.float32):
    return numpy_helper.from_array(np.asarray(value,dtype=dtype),name=name+'_value')
def C(name,value,dtype=np.float32):
    nodes.append(helper.make_node('Constant',[],[name],value=tensor(name,value,dtype)))
    return name
def N(op,ins,outs,**kw): nodes.append(helper.make_node(op,ins,outs,**kw))

C('zero',0.0); C('large',1000.0)
C('axis_h',np.array(2,np.int64),np.int64)
C('axis_w',np.array(3,np.int64),np.int64)
C('axes_ch',np.array([1],np.int64),np.int64)
C('unsq_target',np.array([4],np.int64),np.int64)
C('unsq_h_source',np.array([3],np.int64),np.int64)
C('unsq_v_source',np.array([2],np.int64),np.int64)
C('reduce_pair',np.array([4],np.int64),np.int64)
C('color1',np.eye(10,dtype=np.float32)[1].reshape(1,10,1,1))
C('color8',np.eye(10,dtype=np.float32)[8].reshape(1,10,1,1))

'color8'

In [5]:
# Active canvas and the fixed ARC colors used by this task.
N('ReduceSum',['input','axes_ch'],['active_sum'],keepdims=1)
N('Greater',['active_sum','zero'],['active'])
N('Slice',['input','c0_starts','c0_ends','c0_axes'],['channel0'])
N('Slice',['input','c2_starts','c2_ends','c0_axes'],['channel2'])
N('Greater',['channel0','zero'],['is_zero'])
N('Greater',['channel2','zero'],['is_wall'])
N('Cast',['is_wall'],['wall_f'],to=TensorProto.FLOAT)


In [6]:
# Prefix wall counts label horizontal and vertical wall-delimited runs.
N('CumSum',['wall_f','axis_w'],['h_prefix'])
N('Unsqueeze',['h_prefix','unsq_target'],['h_target'])
N('Unsqueeze',['h_prefix','unsq_h_source'],['h_source'])
N('Equal',['h_target','h_source'],['h_same_segment'])
N('Unsqueeze',['is_zero','unsq_h_source'],['h_zero_source'])
N('And',['h_same_segment','h_zero_source'],['h_members'])
N('Cast',['h_members'],['h_members_f'],to=TensorProto.FLOAT)
N('ReduceSum',['h_members_f','reduce_pair'],['run_width'],keepdims=0)

N('CumSum',['wall_f','axis_h'],['v_prefix'])
N('Unsqueeze',['v_prefix','unsq_target'],['v_target'])
N('Transpose',['v_prefix'],['v_prefix_t'],perm=[0,1,3,2])
N('Unsqueeze',['v_prefix_t','unsq_v_source'],['v_source'])
N('Equal',['v_target','v_source'],['v_same_segment'])
N('Transpose',['is_zero'],['zero_t'],perm=[0,1,3,2])
N('Unsqueeze',['zero_t','unsq_v_source'],['v_zero_source'])
N('And',['v_same_segment','v_zero_source'],['v_members'])
N('Cast',['v_members'],['v_members_f'],to=TensorProto.FLOAT)
N('ReduceSum',['v_members_f','reduce_pair'],['run_height'],keepdims=0)

In [7]:
# Generator components are rectangles: area = row run × column run.
N('Mul',['run_width','run_height'],['component_area_raw'])
N('Cast',['is_zero'],['is_zero_f'],to=TensorProto.FLOAT)
N('Mul',['component_area_raw','is_zero_f'],['component_area'])
N('ReduceMax',['component_area'],['max_area'],axes=[2,3],keepdims=1)
N('Where',['is_zero','component_area','large'],['area_or_large'])
N('ReduceMin',['area_or_large'],['min_area'],axes=[2,3],keepdims=1)
N('Equal',['component_area','max_area'],['largest0'])
N('Equal',['component_area','min_area'],['smallest0'])
N('And',['largest0','is_zero'],['largest'])
N('And',['smallest0','is_zero'],['smallest'])
N('Where',['smallest','color8','input'],['with_smallest'])
N('Where',['largest','color1','with_smallest'],['canvas_output'])
N('Cast',['active'],['active_f'],to=TensorProto.FLOAT)
N('Mul',['canvas_output','active_f'],['output'])

In [8]:
# Slice constants are explicit Constant nodes, avoiding notebook initializer state.
for name,val in [('c0_starts',[0]),('c0_ends',[1]),('c2_starts',[2]),('c2_ends',[3]),('c0_axes',[1])]:
    C(name,np.array(val,np.int64),np.int64)

# Constants must precede their consumers for strict topological checking.
constants=[n for n in nodes if n.op_type=='Constant']
compute=[n for n in nodes if n.op_type!='Constant']
nodes=constants+compute
graph=helper.make_graph(nodes,'task145_rectangular_component_rank_v1',
    [helper.make_tensor_value_info('input',TensorProto.FLOAT,[1,10,30,30])],
    [helper.make_tensor_value_info('output',TensorProto.FLOAT,[1,10,30,30])])
model=helper.make_model(graph,opset_imports=[helper.make_opsetid('',17)],producer_name='task145-v1')
model.ir_version=8
onnx.checker.check_model(model);onnx.save(model,MODEL)

In [9]:
def pad(grid):
    g=np.asarray(grid,np.int64);h,w=g.shape
    x=np.zeros((1,10,30,30),np.float32)
    rr,cc=np.indices((h,w));x[0,g,rr,cc]=1
    return x

data=json.loads(TASK.read_text())
opts=ort.SessionOptions();opts.intra_op_num_threads=1;opts.inter_op_num_threads=1
sess=ort.InferenceSession(str(MODEL),sess_options=opts,providers=['CPUExecutionProvider'])
def run(g):return sess.run(None,{'input':pad(g)})[0]
def validate(cases):return sum(np.array_equal(run(e['input']),pad(e['output'])) for e in cases)
results={}
for split,cases in data.items():
    ok=validate(cases);results[split]={'ok':ok,'total':len(cases)};assert ok==len(cases),(split,ok)
arc=data['arc-gen'];results['arc_gen_dev']={'ok':validate(arc[:104]),'total':104}
results['arc_gen_holdout_60pct']={'ok':validate(arc[104:]),'total':158}


In [10]:
# Published finalist programs: first place, second place, and SakanaAI.
sys.setrecursionlimit(100000)
src1="p=lambda g,n=7,l=1:-n*g or p([(a:=0)or[a:=[n and c|a|(l:=l<<9)or(X:=sorted({x%511for x in sum(g,r)}))[1]//(C:=c%511)*8+C//X[-1],2][c==2]for c in r]for*r,in zip(*g[::-1])],n-1,0)"
src2="p=lambda g,k=6,i=1,q=0:~k*g or p([[q:=[i:=i<<9,(c>(l:=sorted({*sum(g,r)}))[-2])+(c==l[1])*8,2,c%511,0,c|q,0][179%(k-7)+c%~c]for c in[0]+r][:0:-1]for*r,in zip(*g)],k-1)"
src3="def p(a):\n A=len(a[0]);F=len(a)*A;G,D=0,3;B=lambda i:i<F and(C:=a[i//A])[j:=i%A]==G and(C.__setitem__(j,D)or-~B(i+A)+B(i+(A>j+1)));E=[(C,A)for A in range(F)if(C:=B(A))];G=D\n for(C,H)in E:D=C==max(E)[0]or(C==min(E)[0])*8;B(H)\n return a"
solvers=[]
for src in (src1,src2,src3):ns={};exec(src,ns);solvers.append(ns['p'])
official_consensus=0
for cases in data.values():
    for e in cases:
        outs=[p([r[:] for r in e['input']]) for p in solvers]
        assert outs[0]==outs[1]==outs[2]==e['output'];official_consensus+=1
results['published_finalist_consensus']={'ok':official_consensus,'total':official_consensus}


<string>:1: SyntaxWarning: invalid decimal literal


In [11]:
def random_case(rng):
    # Exact ARC-GEN family: recursive guillotine bisection into rectangles.
    while True:
        h=rng.randint(10,20);w=rng.randint(10,20)
        g=[[0]*w for _ in range(h)];leaves=[]
        def cut(r,c,hh,ww,depth=0):
            choices=[0]+([1] if hh>2 else [])+([2] if ww>2 else [])
            kind=rng.choice(choices)
            if kind==0 or depth>12:
                leaves.append((r,c,hh,ww));return
            if kind==1:
                q=rng.randint(r+1,r+hh-2)
                cut(r,c,q-r,ww,depth+1);cut(q+1,c,r+hh-q-1,ww,depth+1)
                for j in range(c,c+ww):g[q][j]=2
            else:
                q=rng.randint(c+1,c+ww-2)
                cut(r,c,hh,q-c,depth+1);cut(r,q+1,hh,c+ww-q-1,depth+1)
                for i in range(r,r+hh):g[i][q]=2
        cut(0,0,h,w)
        areas=[hh*ww for _,_,hh,ww in leaves]
        if min(areas)<max(areas):break
    out=[row[:] for row in g];mn,mx=min(areas),max(areas)
    for (r,c,hh,ww),a in zip(leaves,areas):
        color=8 if a==mn else 1 if a==mx else 0
        if color:
            for i in range(r,r+hh):
                for j in range(c,c+ww):out[i][j]=color
    return g,out

rng=random.Random(145);stress=500
for _ in range(stress):
    g,expected=random_case(rng)
    outs=[p([r[:] for r in g]) for p in solvers]
    assert outs[0]==outs[1]==outs[2]==expected
    h,w=len(g),len(g[0]);pred=run(g).argmax(1)[0,:h,:w].tolist()
    assert pred==expected
results['generated_finalist_consensus_stress']={'ok':stress,'total':stress}


In [12]:
proto=onnx.load(MODEL);ops=sorted({n.op_type for n in proto.graph.node})
forbidden=sorted(set(ops)&{'Loop','Scan','NonZero','Unique','Script','Function'})
assert not forbidden and not proto.functions and MODEL.stat().st_size<1_400_000
assert np.count_nonzero(sess.run(None,{'input':np.zeros((1,10,30,30),np.float32)})[0])==0
summary={'task_id':'task145','arc_id':'6455b5f5','model_family':'rectangular_component_area_rank_finalist_consensus_v1',
 'results':results,'onnx_size_bytes':MODEL.stat().st_size,'onnx_sha256':hashlib.sha256(MODEL.read_bytes()).hexdigest(),
 'node_count':len(proto.graph.node),'constant_nodes':sum(n.op_type=='Constant' for n in proto.graph.node),
 'initializer_count':len(proto.graph.initializer),'ops':ops,'forbidden_ops':forbidden,
 'input_shape':[1,10,30,30],'output_shape':[1,10,30,30],'zip_members':['task145.onnx']}
(OUT/'task145_v1_validation_summary.json').write_text(json.dumps(summary,indent=2))
for zp in (OUT/'submission.zip',OUT/'task145_submission_v1_finalist_consensus.zip'):
    with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:z.write(MODEL,'task145.onnx')
    assert zipfile.ZipFile(zp).namelist()==['task145.onnx']
print(json.dumps(summary,indent=2))


{
  "task_id": "task145",
  "arc_id": "6455b5f5",
  "model_family": "rectangular_component_area_rank_finalist_consensus_v1",
  "results": {
    "train": {
      "ok": 4,
      "total": 4
    },
    "test": {
      "ok": 1,
      "total": 1
    },
    "arc-gen": {
      "ok": 262,
      "total": 262
    },
    "arc_gen_dev": {
      "ok": 104,
      "total": 104
    },
    "arc_gen_holdout_60pct": {
      "ok": 158,
      "total": 158
    },
    "published_finalist_consensus": {
      "ok": 267,
      "total": 267
    },
    "generated_finalist_consensus_stress": {
      "ok": 500,
      "total": 500
    }
  },
  "onnx_size_bytes": 3118,
  "onnx_sha256": "e18869276257858451744a4c9ec6a78d26bb66c5156ab3468605ddeefa854812",
  "node_count": 55,
  "constant_nodes": 16,
  "initializer_count": 0,
  "ops": [
    "And",
    "Cast",
    "Constant",
    "CumSum",
    "Equal",
    "Greater",
    "Mul",
    "ReduceMax",
    "ReduceMin",
    "ReduceSum",
    "Slice",
    "Transpose",
    "Unsqueeze",

In [13]:
import shutil,zipfile
shutil.copy2(OUT/'submission.zip',ROOT/'submission.zip')
assert zipfile.ZipFile(ROOT/'submission.zip').namelist()==['task145.onnx']
print('READY:',ROOT/'submission.zip',(ROOT/'submission.zip').stat().st_size,'bytes')

READY: /kaggle/working/submission.zip 1136 bytes
